# Plant Care Diagnosis Expert System

You will build a small expert system that analyzes observable plant conditions, identifies likely care problems, and recommends actions. The system is intended for classroom practice. It is not a substitute for advice from a horticulture specialist.

# PART A - KNOWLEDGE ENGINEERING

## Define the problem

The expert system receives five observations:

| Attribute | Allowed values | Meaning |
|---|---|---|
| `soil_moisture` | `dry`, `moist`, `wet` | Current condition of the soil |
| `leaf_color` | `green`, `yellow`, `brown` | Dominant leaf color |
| `light_level` | `low`, `medium`, `high` | Typical light received by the plant |
| `has_leaf_spots` | `True`, `False` | Whether visible spots are present |
| `has_soft_stem` | `True`, `False` | Whether the stem feels unusually soft |

The system should infer one or more possible problems, such as underwatering, overwatering, sunburn, low light, fungal disease, or root rot; one or more recommended actions; an explanation showing which rules fired.

**Observations**

1. **Working memory**: Attributes belong in working memory because they are the current state of the plant that the system is analyzing. These values can change over time as the plant's condition changes.

2. **Knowledge base**: The set of rules and facts that the system uses to make decisions.

3. **Explanation**: This is important because it shows how the system reaches a certain conclusion.

4. **Multiple possible problems**: Can occur because the system has multiple attributes to analyze.

## Represent facts and rules

A **fact** is a statement currently known to be true:

```text
soil_dry
leaves_brown
light_high
```

A **production rule** has conditions and a conclusion:

```text
IF soil_dry AND leaves_brown
THEN problem_underwatering
```

During forward chaining, the system repeatedly fires every applicable rule. A rule is applicable when all of its conditions are present in working memory and its conclusion has not already been derived.

### Knowledge-design worksheet

Before writing code, propose at least six rules. Use precise fact names and avoid conclusions that contradict their conditions.

| Rule | Conditions | Conclusion | Rationale |
|---|---|---|---|
| Example | `soil_dry`, `leaves_brown` | `problem_underwatering` | Dry soil and brown leaves may indicate insufficient water. |
| 1 | `soil_wet`, `leaves_yellow` | `problem_overwatering` | Yellowing leaves combined with excessively wet soil indicates too much water. |
| 2 | `light_high`, `leaves_brown` | `problem_sunburn` | High light exposure coupled with scorched brown leaves points to sunburn. |
| 3 | `light_low`, `leaves_yellow` | `problem_low_light` | Low light environment and yellowing leaves suggests light deprivation. |
| 4 | `leaf_spots_present` | `risk_fungal_infection` | Visible leaf spots indicate a risk of active fungal infection. |
| 5 | `soil_wet`, `soft_stem_present` | `problem_root_rot` | Soggy soil and a soft stem are classic symptoms of advanced root rot. |
| 6 | `soil_moist`, `leaves_green`, `leaf_spots_absent`, `soft_stem_absent` | `problem_healthy` | Moist soil, healthy green leaves, and no other symptoms indicate the plant is doing well. |

# PART B - IMPLEMENTATION

## Define the data structures

Read the following classes before continuing:

- `Rule` stores one production rule.
- `InferenceStep` records why a rule fired.
- `InferenceResult` contains the final working memory and the complete trace.

In [1]:
from dataclasses import dataclass
from typing import Iterable


# frozen=True makes the dataclass immutable, which is useful for 
# representing facts and rules that should not change after creation.
@dataclass(frozen=True)
class Rule:
    """Represent one production rule.

    Args:
        name: Unique identifier used in explanations.
        conditions: Facts that must all be true before the rule can fire.
        conclusion: Fact added to working memory when the rule fires.
        explanation: Human-readable reason for the conclusion.
    """

    name: str
    # frozenset store the conditions as an immutable set, which is useful for checking if all conditions are met.
    conditions: frozenset[str]
    conclusion: str
    explanation: str


@dataclass(frozen=True)
class InferenceStep:
    """Record one successful rule application."""

    rule_name: str
    matched_conditions: tuple[str, ...]
    conclusion: str
    explanation: str


# Freezing the dataclass makes it immutable, which is useful for 
# storing inference results that should not change after creation.
@dataclass(frozen=True)
class InferenceResult:
    """Store the facts and explanation trace produced by inference."""

    facts: frozenset[str]
    trace: tuple[InferenceStep, ...]

## Task 4. Implement forward chaining

Behaviors:

1. Copy the initial facts into a mutable working-memory set.
2. Examine every rule in each pass.
3. Fire a rule when all its conditions are present.
4. Add only new conclusions.
5. Record every fired rule in the trace.
6. Stop when a complete pass adds no new facts.

The method must terminate even when rules contain cycles.

In [2]:
class ExpertSystem:
    """Implement a minimal forward-chaining expert system."""

    def __init__(self, rules: Iterable[Rule]):
        """Initialize the system with an ordered collection of rules.

        Args:
            rules: Production rules used by the inference engine.
        """
        self.rules = tuple(rules)

    def infer(self, initial_facts: Iterable[str]) -> InferenceResult:
        """Derive all reachable facts from the supplied facts.

        Args:
            initial_facts: Observations available before reasoning starts.

        Returns:
            An inference result containing final facts and fired rules.
        """
        # Create a mutable set for working memory and a list to store the inference trace.
        # Iterate through the rules until no new facts are added in a pass.
        facts = set(initial_facts)
        trace: list[InferenceStep] = []
        fired_rules: set[str] = set()

        while True:
            new_fact_added = False
            for rule in self.rules:
                if rule.name in fired_rules:
                    continue

                condition_met = rule.conditions.issubset(facts)
                conclusion_is_new = rule.conclusion not in facts

                if condition_met and conclusion_is_new:
                    facts.add(rule.conclusion)
                    fired_rules.add(rule.name)

                    new_fact_added = True
                    trace.append(
                        InferenceStep(
                            rule_name=rule.name,
                            matched_conditions=rule.conditions,
                            conclusion=rule.conclusion,
                            explanation=rule.explanation,
                        )
                    )

            if not new_fact_added:
                break

        return InferenceResult(facts=facts, trace=trace)

### Checkpoint 1 - Test the inference engine

The expected derived facts are:

```text
risk_water_stress
problem_underwatering
action_water_thoroughly
```

In [3]:
ENGINE_TEST_RULES = [
    Rule(
        name="T01",
        conditions=frozenset({"soil_dry"}),
        conclusion="risk_water_stress",
        explanation="Dry soil creates a risk of water stress.",
    ),
    Rule(
        name="T02",
        conditions=frozenset({"risk_water_stress", "leaves_brown"}),
        conclusion="problem_underwatering",
        explanation="Water stress with brown leaves suggests underwatering.",
    ),
    Rule(
        name="T03",
        conditions=frozenset({"problem_underwatering"}),
        conclusion="action_water_thoroughly",
        explanation="An underwatered plant should be watered thoroughly.",
    ),
]

# The three rules above are designed to be triggered in order, with each rule's conclusion serving as a condition for the next rule. 
# This setup allows us to test the forward-chaining inference engine's ability to derive new facts based on initial observations.
checkpoint_1 = ExpertSystem(ENGINE_TEST_RULES).infer({"soil_dry", "leaves_brown"})

expected_1 = {
    "soil_dry",
    "leaves_brown",
    "risk_water_stress",
    "problem_underwatering",
    "action_water_thoroughly",
}
assert checkpoint_1.facts == expected_1
# List comprehension to extract rule names from the inference trace and compare with expected order.
assert [step.rule_name for step in checkpoint_1.trace] == ["T01", "T02", "T03"]
print("Checkpoint 1 passed.")

Checkpoint 1 passed.


## Task 5. Build the knowledge base

The starter knowledge base below is incomplete. Add rules until the list contains at least **16 rules**.

Your completed knowledge base must be able to infer all of these problem categories:

- `problem_underwatering`
- `problem_overwatering`
- `problem_sunburn`
- `problem_low_light`
- `problem_fungal_disease`
- `problem_root_rot`
- `problem_healthy`

It must also infer at least one `action_...` fact for each problem.

In [4]:
RULES = [
    # Rules for diagnosing plant health based on soil moisture, leaf color, and other observable conditions.
    Rule(
        name="R01",
        conditions=frozenset({"soil_dry"}),
        conclusion="risk_water_stress",
        explanation="The soil is dry, so the plant may be experiencing water stress.",
    ),
    Rule(
        name="R02",
        conditions=frozenset({"soil_wet", "leaves_yellow"}),
        conclusion="problem_overwatering",
        explanation="Wet soil together with yellow leaves suggests overwatering.",
    ),
    Rule(
        name="R03",
        conditions=frozenset({"light_high", "leaves_brown"}),
        conclusion="problem_sunburn",
        explanation="Brown leaves under intense light suggest sunburn.",
    ),
    Rule(
        name="R04",
        conditions=frozenset({"leaf_spots_present"}),
        conclusion="risk_fungal_infection",
        explanation="Visible leaf spots may indicate a fungal infection.",
    ),
    Rule(
        name="R05",
        conditions=frozenset({"soil_wet", "soft_stem_present"}),
        conclusion="problem_root_rot",
        explanation="Wet soil and a soft stem are warning signs of root rot.",
    ),
    # Rules for diagnosing a healthy plant based on multiple positive indicators.
    Rule(
        name="R06",
        conditions=frozenset({"soil_moist", "leaves_green", "leaf_spots_absent", "soft_stem_absent"}),
        conclusion="problem_healthy",
        explanation="Moist soil, green leaves, and no visible damage suggest a healthy plant.",
    ),
    
    # Some additional rules to cover more scenarios and actions.
    Rule(
        name="R07",
        conditions=frozenset({"risk_water_stress", "leaves_brown"}),
        conclusion="problem_underwatering",
        explanation="Water stress combined with brown leaves indicates underwatering.",
    ),
    Rule(
        name="R08",
        conditions=frozenset({"light_low", "leaves_yellow"}),
        conclusion="problem_low_light",
        explanation="Low light and yellowing leaves indicate the plant is not receiving enough light.",
    ),
    Rule(
        name="R09",
        conditions=frozenset({"risk_fungal_infection", "leaves_yellow"}),
        conclusion="problem_fungal_disease",
        explanation="Fungal infection indicators combined with yellow leaves suggest fungal disease.",
    ),
    Rule(
        name="R10",
        conditions=frozenset({"problem_underwatering"}),
        conclusion="action_water_thoroughly",
        explanation="If underwatered, water the plant thoroughly and immediately.",
    ),
    Rule(
        name="R11",
        conditions=frozenset({"problem_overwatering"}),
        conclusion="action_reduce_watering",
        explanation="If overwatered, allow the soil to dry out and reduce watering frequency.",
    ),
    Rule(
        name="R12",
        conditions=frozenset({"problem_sunburn"}),
        conclusion="action_move_to_shade",
        explanation="If sunburned, move the plant away from direct sunlight to a shadier spot.",
    ),
    Rule(
        name="R13",
        conditions=frozenset({"problem_low_light"}),
        conclusion="action_increase_light",
        explanation="If suffering from low light, move the plant closer to a window or add artificial light.",
    ),
    Rule(
        name="R14",
        conditions=frozenset({"problem_fungal_disease"}),
        conclusion="action_apply_fungicide",
        explanation="If fungal disease is detected, treat the foliage with an appropriate fungicide.",
    ),
    Rule(
        name="R15",
        conditions=frozenset({"problem_root_rot"}),
        conclusion="action_repot_plant",
        explanation="If root rot has set in, repot the plant in fresh soil and trim damaged roots.",
    ),
    Rule(
        name="R16",
        conditions=frozenset({"problem_healthy"}),
        conclusion="action_maintain_care",
        explanation="If the plant is healthy, maintain your current care routine.",
    )
]

### Checkpoint 2 - Validate knowledge-base coverage

This check validates structure and coverage; it does not prove that every rule is horticulturally correct.

In [5]:
REQUIRED_PROBLEMS = {
    "problem_underwatering",
    "problem_overwatering",
    "problem_sunburn",
    "problem_low_light",
    "problem_fungal_disease",
    "problem_root_rot",
    "problem_healthy",
}

# Collect rule names, conclusions, and action rules for validation checks.
rule_names = [rule.name for rule in RULES]
conclusions = {rule.conclusion for rule in RULES}
action_rules = [rule for rule in RULES if rule.conclusion.startswith("action_")]

assert len(RULES) >= 16, "Add more rules to the knowledge base."
assert len(rule_names) == len(set(rule_names)), "Every rule name must be unique."
# Check that all required problem conclusions are present in the rules.
assert REQUIRED_PROBLEMS <= conclusions, "Some required problem conclusions are missing."
assert len(action_rules) >= len(REQUIRED_PROBLEMS), "Add at least one action rule per problem."

print(f"Checkpoint 2 passed with {len(RULES)} rules.")

Checkpoint 2 passed with 16 rules.


## Task 6. Convert observations into facts

Complete `observations_to_facts()`.

Example input:

```python
{
    "soil_moisture": "dry",
    "leaf_color": "brown",
    "light_level": "high",
    "has_leaf_spots": False,
    "has_soft_stem": False,
}
```

Expected facts:

```text
soil_dry, leaves_brown, light_high, leaf_spots_absent, soft_stem_absent
```

Raise `ValueError` when a required field is missing, a category is invalid, or a Boolean field is not a real `bool` value.

In [6]:
VALID_SOIL_MOISTURE = {"dry", "moist", "wet"}
VALID_LEAF_COLORS = {"green", "yellow", "brown"}
VALID_LIGHT_LEVELS = {"low", "medium", "high"}


def observations_to_facts(observations: dict) -> set[str]:
    """Validate plant observations and convert them into symbolic facts.

    Args:
        observations: Mapping containing all five required observations.

    Returns:
        A set of normalized facts for the inference engine.

    Raises:
        ValueError: If a field is missing or contains an invalid value.
    """
    # Check for missing fields
    soil = observations.get("soil_moisture")
    if soil not in VALID_SOIL_MOISTURE:
        raise ValueError(f"Invalid soil moisture: {soil}")

    leaf_color = observations.get("leaf_color")
    if leaf_color not in VALID_LEAF_COLORS:
        raise ValueError(f"Invalid leaf color: {leaf_color}")

    light_level = observations.get("light_level")
    if light_level not in VALID_LIGHT_LEVELS:
        raise ValueError(f"Invalid light level: {light_level}")

    # Use type(value) is bool to ensure that only boolean values are accepted for binary symptoms.
    has_leaf_spots = observations.get("has_leaf_spots")
    if type(has_leaf_spots) is not bool:
        raise ValueError(f"Invalid leaf spots: {has_leaf_spots}")

    has_soft_stem = observations.get("has_soft_stem")
    if type(has_soft_stem) is not bool:
        raise ValueError(f"Invalid soft stem: {has_soft_stem}")

    # Normalize observations into symbolic facts for the inference engine.
    facts = set()

    # Map soil moisture
    if soil == "dry":
        facts.add("soil_dry")
    elif soil == "moist":
        facts.add("soil_moist")
    elif soil == "wet":
        facts.add("soil_wet")

    # Map leaf color
    if leaf_color == "brown":
        facts.add("leaves_brown")
    elif leaf_color == "yellow":
        facts.add("leaves_yellow")
    elif leaf_color == "green":
        facts.add("leaves_green")

    # Map binary symptoms
    if has_leaf_spots:
        facts.add("leaf_spots_present")
    else:
        facts.add("leaf_spots_absent")

    if has_soft_stem:
        facts.add("soft_stem_present")
    else:
        facts.add("soft_stem_absent")

    # Map light levels
    if light_level == "high":
        facts.add("light_high")
    elif light_level == "medium":
        facts.add("light_medium")
    elif light_level == "low":
        facts.add("light_low")

    return facts

### Checkpoint 3 - Test input normalization

In [7]:
checkpoint_3 = observations_to_facts(
    {
        "soil_moisture": "dry",
        "leaf_color": "brown",
        "light_level": "high",
        "has_leaf_spots": False,
        "has_soft_stem": False,
    }
)

# Compare the output of observations_to_facts with the expected set of facts to ensure correctness.
assert checkpoint_3 == {
    "soil_dry",
    "leaves_brown",
    "light_high",
    "leaf_spots_absent",
    "soft_stem_absent",
}
print("Checkpoint 3 passed.")

Checkpoint 3 passed.


## Task 7. Build the diagnosis result

The function should:

1. Collect every fact beginning with `problem_`.
2. Collect every fact beginning with `action_`.
3. Sort both collections to make the output deterministic.
4. Raise a clear error when no problem or no action can be inferred.

Unlike a single-answer classifier, this expert system may report multiple possible problems when several rule sets are satisfied.

In [8]:
@dataclass(frozen=True)
class Diagnosis:
    """Contain inferred plant problems, recommended actions, and reasoning.

    Attributes:
        problems: Symbolic problem facts inferred by the system.
        actions: Symbolic action facts recommended by the system.
        trace: Ordered rule applications supporting the result.
    """

    problems: tuple[str, ...]
    actions: tuple[str, ...]
    trace: tuple[InferenceStep, ...]


def build_diagnosis(result: InferenceResult) -> Diagnosis:
    """Extract problems and actions from an inference result.

    Args:
        result: Completed forward-chaining result.

    Returns:
        A structured diagnosis with its explanation trace.

    Raises:
        RuntimeError: If no problem or action is available.
    """
    # Filter the inferred facts to separate problems and actions, ensuring they are sorted for consistency.
    # sorted() is used to maintain a consistent order for testing and output purposes.
    problems = tuple(sorted(fact for fact in result.facts if fact.startswith("problem_")))
    actions = tuple(sorted(fact for fact in result.facts if fact.startswith("action_")))

    if not problems:
        raise RuntimeError("No problems were inferred.")
    if not actions:
        raise RuntimeError("No actions were inferred.")

    return Diagnosis(problems=problems, actions=actions, trace=result.trace)

## Task 8. Integrate the system and print an explanation

Both functions are used to run the complete expert system. The first function converts observations into facts, and the second function runs the inference engine and builds the diagnosis result.

The displayed explanation must show: initial observations; fired rule names; matched conditions; newly derived conclusions; final problems and actions.

In [9]:
plant_expert_system = ExpertSystem(RULES)


def diagnose_plant(
    observations: dict,
    system: ExpertSystem = plant_expert_system,
) -> Diagnosis:
    """Run the complete plant-care consultation workflow.

    Args:
        observations: Raw plant observations supplied by the user.
        system: Expert system used to perform inference.

    Returns:
        A diagnosis containing problems, actions, and a rule trace.
    """
    # Merge the observation validation and fact conversion into a single step, 
    # then run the inference engine and build the diagnosis.
    facts = observations_to_facts(observations)
    result = system.infer(facts)
    diagnosis = build_diagnosis(result)
    return diagnosis


def print_diagnosis(observations: dict, diagnosis: Diagnosis) -> None:
    """Print a readable diagnosis and explanation.

    Args:
        observations: Raw observations supplied to the system.
        diagnosis: Result returned by diagnose_plant().
    """
    # Traverse diagnosis.trace to explain each fired rule, joining matched_conditions with ' AND ' 
    # to show the structure of the rule.
    print("Initial observations:")
    for key, value in observations.items():
        print(f"- {key}: {value}")

    print("\nRule trace:")
    for step in diagnosis.trace:
        print(f"- {step.rule_name}:")
        print(f"  conditions: { ' AND '.join(step.matched_conditions) }")
        print(f"  conclusion: {step.conclusion}")
        print(f"  explanation: {step.explanation}")

    print("\nProblems:")
    for problem in diagnosis.problems:
        print(f"- {problem}")

    print("\nActions:")
    for action in diagnosis.actions:
        print(f"- {action}")

### Checkpoint 4 - Run a complete consultation

Complete Tasks 4-8 before running the next cell. The exact result depends on your rules, but the output should include an underwatering or sunburn-related conclusion and at least one action.

In [10]:
sample_observations = {
    "soil_moisture": "dry",
    "leaf_color": "brown",
    "light_level": "high",
    "has_leaf_spots": False,
    "has_soft_stem": False,
}

sample_diagnosis = diagnose_plant(sample_observations)
print_diagnosis(sample_observations, sample_diagnosis)

# The assert statements below are used to validate that the diagnosis contains problems, actions, and a trace of inference steps.
assert sample_diagnosis.problems
assert sample_diagnosis.actions
assert sample_diagnosis.trace
print("Checkpoint 4 passed.")

Initial observations:
- soil_moisture: dry
- leaf_color: brown
- light_level: high
- has_leaf_spots: False
- has_soft_stem: False

Rule trace:
- R01:
  conditions: soil_dry
  conclusion: risk_water_stress
  explanation: The soil is dry, so the plant may be experiencing water stress.
- R03:
  conditions: leaves_brown AND light_high
  conclusion: problem_sunburn
  explanation: Brown leaves under intense light suggest sunburn.
- R07:
  conditions: leaves_brown AND risk_water_stress
  conclusion: problem_underwatering
  explanation: Water stress combined with brown leaves indicates underwatering.
- R10:
  conditions: problem_underwatering
  conclusion: action_water_thoroughly
  explanation: If underwatered, water the plant thoroughly and immediately.
- R12:
  conditions: problem_sunburn
  conclusion: action_move_to_shade
  explanation: If sunburned, move the plant away from direct sunlight to a shadier spot.

Problems:
- problem_sunburn
- problem_underwatering

Actions:
- action_move_to_sh

# PART C - TESTING AND EVALUATION

## Task 9. Build a test suite

The starter cases below cover four common situations. Add at least four more cases so your suite contains **8 or more scenarios**.

Good additional cases include: a healthy plant; low-light stress; simultaneous fungal and watering problems; a boundary case that fires only one problem rule.

In [11]:
TEST_CASES = [
    (
        "Underwatering",
        {
            "soil_moisture": "dry",
            "leaf_color": "brown",
            "light_level": "medium",
            "has_leaf_spots": False,
            "has_soft_stem": False,
        },
        "problem_underwatering",
    ),
    (
        "Overwatering",
        {
            "soil_moisture": "wet",
            "leaf_color": "yellow",
            "light_level": "medium",
            "has_leaf_spots": False,
            "has_soft_stem": False,
        },
        "problem_overwatering",
    ),
    (
        "Fungal disease",
        {
            "soil_moisture": "moist",
            "leaf_color": "yellow",
            "light_level": "medium",
            "has_leaf_spots": True,
            "has_soft_stem": False,
        },
        "problem_fungal_disease",
    ),
    (
        "Root rot",
        {
            "soil_moisture": "wet",
            "leaf_color": "yellow",
            "light_level": "low",
            "has_leaf_spots": False,
            "has_soft_stem": True,
        },
        "problem_root_rot",
    ),
    #  Add at least four more test cases to ensure the system can handle a variety of scenarios.
    (
        "Healthy Plant",
        {
            "soil_moisture": "moist",
            "leaf_color": "green",
            "light_level": "medium",
            "has_leaf_spots": False,
            "has_soft_stem": False,
        },
        "problem_healthy",
    ),
    (
        "Low-Light Stress",
        {
            "soil_moisture": "moist",
            "leaf_color": "yellow",
            "light_level": "low",
            "has_leaf_spots": False,
            "has_soft_stem": False,
        },
        "problem_low_light",
    ),
    (
        "Fungal and Underwatering Combo",
        {
            "soil_moisture": "dry",
            "leaf_color": "brown",
            "light_level": "low",
            "has_leaf_spots": True,
            "has_soft_stem": False,
        },
        "problem_underwatering",
    ),
    (
        "Boundary Sunburn Only",
        {
            "soil_moisture": "moist",
            "leaf_color": "brown",
            "light_level": "high",
            "has_leaf_spots": False,
            "has_soft_stem": False,
        },
        "problem_sunburn",
    )
]


def run_test_suite(test_cases: list[tuple]) -> None:
    """Execute diagnosis scenarios and assert expected problems.

    Args:
        test_cases: Tuples containing a label, observations, and expected problem.
    """
    assert len(test_cases) >= 8, "Add at least eight test scenarios."

    # Each test case is run through the diagnose_plant function, and the expected problem is checked against the inferred problems.
    for label, observations, expected_problem in test_cases:
        diagnosis = diagnose_plant(observations)
        # Use the 'in' operator because the system may infer multiple problems simultaneously, 
        # and we want to ensure that the expected problem is among them.
        assert expected_problem in diagnosis.problems, (
            f"{label}: expected {expected_problem}, got {diagnosis.problems}"
        )
        print(f"PASS: {label}")


run_test_suite(TEST_CASES)

PASS: Underwatering
PASS: Overwatering
PASS: Fungal disease
PASS: Root rot
PASS: Healthy Plant
PASS: Low-Light Stress
PASS: Fungal and Underwatering Combo
PASS: Boundary Sunburn Only


## Task 10. Test invalid input

Test at least these errors:

1. Missing required field.
2. Invalid category such as `"very_wet"`.
3. Non-Boolean symptom value such as `"yes"`.

In [12]:
INVALID_CASES = [
    {
        "soil_moisture": "very_wet",
        "leaf_color": "yellow",
        "light_level": "low",
        "has_leaf_spots": False,
        "has_soft_stem": True,
    },
    # Add a missing-field case and a non-Boolean case.
    {
        "soil_moisture": "moist",
        "leaf_color": "green",
        "light_level": "medium",
        # Missing 'has_leaf_spots' field
        "has_soft_stem": False,
    },
    {
        "soil_moisture": "dry",
        "leaf_color": "brown",
        "light_level": "high",
        "has_leaf_spots": "yes",  # Non-Boolean value
        "has_soft_stem": False,
    },
]

# The trial below tests the system's ability to reject invalid input by raising ValueError exceptions. 
# Each invalid case is processed, and the expected exception is caught and reported.
for invalid_observations in INVALID_CASES:
    try:
        observations_to_facts(invalid_observations)
    except ValueError as error:
        print(f"PASS: rejected invalid input ({error})")
    else:
        raise AssertionError("Invalid input was accepted.")

PASS: rejected invalid input (Invalid soil moisture: very_wet)
PASS: rejected invalid input (Invalid leaf spots: None)
PASS: rejected invalid input (Invalid leaf spots: yes)


## Task 11. Analyze the system

The system should be able to answer these questions:

1. **Initial facts**: `soil_dry`, `leaves_brown`, `light_high`, `leaf_spots_absent`, `soft_stem_absent`.
2. **Derived facts**: `risk_water_stress`, `problem_sunburn`, `problem_underwatering`, `action_water_thoroughly`, `action_move_to_shade`.
3. **Final problem rules**: `R03` (sunburn) and `R07` (underwatering).
4. **Rules in the same pass**: Yes, multiple independent rules can fire in the same pass if their conditions are met simultaneously (e.g., `R01` and `R03`).
5. **Termination**: The `while True` loop terminates when a full pass over all rules produces no new conclusions (`new_fact_added = False`).
6. **Limitation**: The system can't distinguish between a temporary dry spell and chronic underwatering if given only a single point-in-time observation.
7. **Monotonicity**: Yes, the reasoning is monotonic. The size of working memory `facts` only grows; once a fact is derived, it is never removed or contradicted.

## Recap

Building this rule-based expert system provided valuable insights into the separation of domain knowledge and inference engine. The most challenging aspect was ensuring the logical progression from initial observations to intermediate states and finally to concrete diagnoses and corresponding care actions without introducing logic cycles or unreachable rules.

Testing with multiple test cases helped uncover critical bugs in our input normalization mapping, forcing us to account for all states of categorical values rather than just the positive symptoms. This highlighted how fragile rule-based engines can be if the translator function does not properly align observations with working memory variables.

Two current limitations include the inability to process uncertain or fuzzy values (such as 'slightly dry' soil) and the lack of temporal context, meaning the system cannot track changes in plant health over time. If more expert knowledge became available, we could implement probabilistic reasoning like certainty factors or Bayesian belief networks to handle nuanced, non-binary scenarios.